# Synthetic Disease Risk Prediction — Exploratory Data Analysis

This notebook explores the synthetic disease risk dataset used in the
`Synthetic Disease Risk Prediction` Streamlit application.

**Note:** This is a synthetic, educational dataset. Nothing in this notebook
constitutes medical advice or a diagnostic tool.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import load_dataset, clean_dataset

sns.set_style('whitegrid')
pd.set_option('display.max_columns', None)


## 1. Load the dataset

In [ ]:
DATA_PATH = os.path.join('..', 'data', 'Synthetic_disease_risk_dataset.csv')
raw_df = load_dataset(DATA_PATH)
print('Raw shape:', raw_df.shape)
raw_df.head()


## 2. Data quality checks

In [ ]:
print('Missing values per column:')
print(raw_df.isnull().sum())
print()
print('Duplicate rows:', raw_df.duplicated().sum())
print()
print('Data types:')
print(raw_df.dtypes)


In [ ]:
df = clean_dataset(raw_df)
df.describe(include='all').transpose()


## 3. Target variable distribution

In [ ]:
risk_counts = df['Disease_Risk'].value_counts()
print(risk_counts)
print()
print('Positive class rate: {:.2f}%'.format(100 * risk_counts['Yes'] / len(df)))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='Disease_Risk', hue='Disease_Risk',
              palette={'No': '#28a745', 'Yes': '#d9534f'}, legend=False)
plt.title('Disease Risk Distribution')
plt.show()


## 4. Numeric feature distributions

In [ ]:
numeric_cols = ['Age', 'BMI', 'Blood_Pressure_Systolic', 'Blood_Pressure_Diastolic',
                'Cholesterol_Level', 'Glucose_Level', 'Genetic_Risk_Score']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue='Disease_Risk', kde=True, ax=axes[i],
                 palette={'No': '#28a745', 'Yes': '#d9534f'}, element='step')
    axes[i].set_title(f'{col} Distribution')
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()


## 5. Categorical features vs Disease Risk

In [ ]:
categorical_cols = ['Gender', 'Smoking_Status', 'Alcohol_Consumption',
                    'Physical_Activity_Level', 'Family_History', 'Previous_Diagnosis']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(categorical_cols):
    ct = pd.crosstab(df[col], df['Disease_Risk'], normalize='index') * 100
    ct.plot(kind='bar', stacked=False, ax=axes[i], color=['#28a745', '#d9534f'])
    axes[i].set_title(f'{col} vs Disease Risk (% within group)')
    axes[i].set_ylabel('% of group')
    axes[i].legend(title='Disease Risk')
plt.tight_layout()
plt.show()


## 6. Correlation among numeric features

In [ ]:
corr_df = df[numeric_cols].copy()
corr_df['Disease_Risk'] = df['Disease_Risk'].map({'Yes': 1, 'No': 0})

plt.figure(figsize=(8, 6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix (numeric features + target)')
plt.show()


## 7. Key observations

- The target variable `Disease_Risk` is imbalanced: roughly 14–15% of patients are
  labeled "Yes" (at risk).
- `Alcohol_Consumption` and `Previous_Diagnosis` contain missing values that
  correspond to a real-world "none reported" category rather than random
  missingness, and are filled accordingly during preprocessing (see
  `src/data_preprocessing.py`).
- Several numeric features (e.g. Blood Pressure, Cholesterol, Glucose, Genetic
  Risk Score) show visibly different distributions between the "Yes" and "No"
  risk groups, suggesting they carry predictive signal — consistent with the
  model evaluation results in `models/model_metrics.json`.

See `PROJECT_REPORT.md` for the full modeling methodology and results.
